In [2]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from joblib import Parallel, delayed
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.io import imread
from skimage.transform import resize
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from tqdm import tqdm

import joblib
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

SEED = 42
IMAGE_SIZE = (128, 128)
HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (6, 6),
    "cells_per_block": (3, 3),
    "block_norm": "L2-Hys",
}
VAL_SIZE = 0.2
N_JOBS = 8
OPTUNA_TRIALS = 30
OPTUNA_JOBS = 4

def resolve_data_dir() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidate = parent / "data"
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/train and data/test")

def collect_paths(split_dir: Path) -> tuple[list[Path], np.ndarray, list[str]]:
    labels = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    if not labels:
        raise RuntimeError(f"No class folders found in {split_dir}")

    paths: list[Path] = []
    y: list[str] = []
    for label in labels:
        for path in sorted((split_dir / label).glob("*.*")):
            if path.is_file():
                paths.append(path)
                y.append(label)

    if not paths:
        raise RuntimeError(f"No images found in {split_dir}")

    return paths, np.array(y), labels

def compute_features(
    paths: list[Path],
    labels: np.ndarray,
    image_size: tuple[int, int],
    hog_params: dict,
    cache_file: Path | None,
    n_jobs: int,
) -> tuple[np.ndarray, np.ndarray]:
    if cache_file is not None and cache_file.exists():
        data = np.load(cache_file)
        return data["X"], data["y"]

    def extract(path: Path) -> np.ndarray | None:
        try:
            img = imread(path)
            gray = rgb2gray(img) if img.ndim == 3 else img
            gray = resize(gray, image_size, anti_aliasing=True)
            return hog(gray, **hog_params)
        except Exception:
            return None

    desc = f"HOG {cache_file.stem}" if cache_file is not None else "HOG"
    features = Parallel(n_jobs=n_jobs)(
        delayed(extract)(p) for p in tqdm(paths, desc=desc, total=len(paths))
    )

    kept_feats: list[np.ndarray] = []
    kept_labels: list[str] = []
    skipped = 0
    for feat, label in zip(features, labels):
        if feat is None:
            skipped += 1
            continue
        kept_feats.append(feat)
        kept_labels.append(label)

    if not kept_feats:
        raise RuntimeError("No features extracted; check input images.")

    X = np.vstack(kept_feats)
    y = np.array(kept_labels)

    if cache_file is not None:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_file, X=X, y=y)

    if skipped:
        print(f"Skipped {skipped} unreadable files.")

    return X, y

DATA_DIR = resolve_data_dir()
MODELS_DIR = DATA_DIR.parent / "models"
CACHE_DIR = DATA_DIR.parent / "cache"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_paths, train_labels, train_classes = collect_paths(DATA_DIR / "train")
test_paths, test_labels, test_classes = collect_paths(DATA_DIR / "test")

if set(train_classes) != set(test_classes):
    print("Warning: train/test class folders differ; using union of labels.")
all_classes = sorted(set(train_classes) | set(test_classes))

train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths,
    train_labels,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_labels,
)

cache_tag = (
    f"hog_svm_6x3_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_"
    f"val{int(VAL_SIZE * 100)}_seed{SEED}"
)
X_train, y_train = compute_features(
    train_paths, y_train, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_train.npz", N_JOBS
)
X_val, y_val = compute_features(
    val_paths, y_val, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_val.npz", N_JOBS
)
X_test, y_test = compute_features(
    test_paths, test_labels, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_test.npz", N_JOBS
)

le = LabelEncoder()
le.fit(all_classes)
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

def objective(trial: optuna.Trial) -> float:
    params = {
        "C": trial.suggest_float("C", 1e-2, 1e2, log=True),
        "kernel": trial.suggest_categorical("kernel", ["linear", "rbf"]),
        "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
        "random_state": SEED,
        "probability": False,
    }
    clf = SVC(**params)
    clf.fit(X_train, y_train_enc)
    y_pred = clf.predict(X_val)
    return accuracy_score(y_val_enc, y_pred)

study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    sampler=TPESampler(multivariate=True),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, n_jobs=OPTUNA_JOBS)

print("Best validation accuracy:", study.best_value)
print("Best params:", study.best_params)

X_combined = np.vstack([X_train, X_val])
y_combined = np.concatenate([y_train_enc, y_val_enc])

best_params = dict(study.best_params)
best_params["random_state"] = SEED
best_params["probability"] = False

best_clf = SVC(**best_params)
best_clf.fit(X_combined, y_combined)

y_pred_test = best_clf.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test_enc, y_pred_test))
print(classification_report(y_test_enc, y_pred_test, target_names=le.classes_))

model_path = MODELS_DIR / "HOG_SVM_6x3.joblib"
payload = {
    "model": best_clf,
    "label_encoder": le,
    "hog_params": HOG_PARAMS,
    "image_size": IMAGE_SIZE,
    "classes": list(le.classes_),
}
joblib.dump(payload, model_path)
print(f"Saved model to {model_path}")

e:\Nam_3_HK2\ThiGiac\DoAn\demo\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
HOG hog_svm_6x3_128x128_val20_seed42_train: 100%|██████████| 1075/1075 [00:09<00:00, 117.46it/s]


Skipped 116 unreadable files.


HOG hog_svm_6x3_128x128_val20_seed42_val: 100%|██████████| 269/269 [00:00<00:00, 393.92it/s]


Skipped 32 unreadable files.


HOG hog_svm_6x3_128x128_val20_seed42_test: 100%|██████████| 155/155 [00:00<00:00, 291.88it/s]
e:\Nam_3_HK2\ThiGiac\DoAn\demo\.venv\lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-05-13 01:01:36,551] A new study created in memory with name: no-name-c368dc62-7a29-47e6-b818-ad00dfdd0de2
[I 2026-05-13 01:02:44,244] Trial 2 finished with value: 0.8312236286919831 and parameters: {'C': 0.05603617802336556, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 2 with value: 0.8312236286919831.
[I 2026-05-13 01:02:55,218] Trial 3 finished with value: 0.729957805907173 and parameters: {'C': 11.03622056172061, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 2 with value: 0.8312236286919831.
[I 2026-05-13 01:03:15,418] Trial 0 finished with value: 0.8481012658227848 and parameters: {'C': 1.918801070885816, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 wit

Best validation accuracy: 0.8481012658227848
Best params: {'C': 1.918801070885816, 'kernel': 'rbf', 'gamma': 'scale'}

Test Accuracy: 0.7870967741935484
              precision    recall  f1-score   support

         Cam       0.84      0.89      0.86        36
      Chidan       0.75      0.75      0.75        36
    Hieulenh       0.68      0.68      0.68        31
    Nguyhiem       1.00      1.00      1.00        29
         Phu       0.62      0.57      0.59        23

    accuracy                           0.79       155
   macro avg       0.78      0.78      0.78       155
weighted avg       0.78      0.79      0.79       155

Saved model to E:\Nam_3_HK2\ThiGiac\DoAn\demo\models\HOG_SVM_6x3.joblib
